# MiniGPT-Stack pretraining launcher (THIN — all logic lives in train/)

One Save Version = one MODE. Set MODE in the next cell, run top to bottom, Save Version.
- `prep`: full tokenizer-data prepare once (streams ~2GB). Output: shards dataset.
- `calibrate`: throughput probe + fp16 gate + kill-resume drill + LR sweep. Output: numbers.
- `main`: resume-latest training, clean exit before the kill. Output: checkpoints.

In [ ]:
MODE = "calibrate"  # prep | calibrate | main  <-- SET THIS, then run all
assert MODE in ("prep", "calibrate", "main"), MODE
import torch
print("MODE", MODE)
print("gpu", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (prep mode)")
print("torch", torch.__version__, "| cuda build", torch.version.cuda,
      "| cuda avail", torch.cuda.is_available())
if MODE != "prep":
    assert torch.cuda.is_available(), "enable GPU in notebook settings"
# Copy these lines into docs/decisions.md with the run results.

In [ ]:
import os
REPO = "https://github.com/Priyanshaur/Glassbox.git"
ROOT = "/kaggle/working/Glassbox"
if not os.path.exists(ROOT):
    !git clone $REPO $ROOT
else:
    !git -C $ROOT pull --ff-only
%cd $ROOT
!git rev-parse --short HEAD

In [ ]:
# Kaggle already ships numpy, pyyaml, tqdm, pytest, regex and datasets. Do NOT install the
# pinned requirements.txt here: it downgrades numpy and rebuilds it from source (~3 min).
!pip install -q regex pyyaml pytest tqdm datasets
!python -m pytest tests/test_config.py tests/test_model.py tests/test_pretrain.py -q 2>&1 | tail -n 2

## Restore previous checkpoints (calibrate/main only)
Attach the previous version's output first (Add Data -> previous run output).

In [ ]:
import glob, os, re, shutil
if MODE not in ("calibrate", "main"):
    print("skip: prep starts fresh")
else:
    cands = [p for p in glob.glob("/kaggle/input/**/ckpt_step*.pt", recursive=True)
             if "nan" not in os.path.basename(p)]
    def _step(p):
        m = re.search(r"ckpt_step(\d+)\.pt", os.path.basename(p))
        return int(m.group(1)) if m else -1
    cands = sorted(cands, key=_step)
    if cands:
        newest, src_dir = cands[-1], os.path.dirname(cands[-1])
        print(f"newest by numeric step: {newest} (step {_step(newest)})")
        dst_dir = os.path.join("checkpoints", os.path.basename(src_dir))
        try:
            import yaml
            cfg_dir = yaml.safe_load(open("configs/base_30m.yaml"))["checkpoint"]["dir"]
            if os.path.normpath(cfg_dir) != os.path.normpath(dst_dir):
                print(f"WARNING: restoring into {dst_dir} but the config resumes from {cfg_dir}; "
                      "--resume latest will NOT find these files")
            else:
                print("restore destination matches config checkpoint dir:", dst_dir)
        except Exception as e:
            print("could not compare with the config checkpoint dir:", e)
        os.makedirs(dst_dir, exist_ok=True)
        for f in glob.glob(os.path.join(src_dir, "*")):
            if os.path.basename(f) not in os.listdir(dst_dir):
                shutil.copy(f, dst_dir)
        print("restored", len(os.listdir(dst_dir)), "files ->", dst_dir)
    else:
        print("no previous checkpoints attached; starting fresh")

## Shards
prep: run the FULL prepare once. calibrate/main: attach the shards dataset and print exact counts.

In [ ]:
import glob, json, os
if MODE == "prep":
    !python -m data.prepare --tokenizer tokenizer/tok8192.json --out_dir data
else:
    hits = glob.glob("/kaggle/input/**/meta.json", recursive=True)
    assert hits, "attach the shards dataset first"
    print("meta.json candidates:", hits)
    shard_dir = os.path.dirname(hits[0])
    print("--- meta.json head ---")
    print("\n".join(open(os.path.join(shard_dir, "meta.json")).read().splitlines()[:15]))
    os.makedirs("data", exist_ok=True)
    # The cloned repo can contain a stale data/meta.json (smoke-test counts). If it is left in
    # place, the symlink below is skipped and training would read the wrong token counts.
    for f in glob.glob("data/*.bin") + ["data/meta.json"]:
        if os.path.lexists(f):
            os.remove(f)
    for f in glob.glob(os.path.join(shard_dir, "*.bin")) + [os.path.join(shard_dir, "meta.json")]:
        os.symlink(f, os.path.join("data", os.path.basename(f)))
    print("linked:", sorted(os.listdir("data")))
    meta = json.load(open("data/meta.json"))
    assert "shards" in meta and "vocab_size" in meta, meta.keys()
    for s in meta["shards"]:
        assert {"split", "index", "file", "tokens"} <= set(s), s
    print("meta keys OK:", sorted(meta.keys()))
    tr = sum(s["tokens"] for s in meta["shards"] if s["split"] == "train")
    va = sum(s["tokens"] for s in meta["shards"] if s["split"] == "validation")
    print(f"EXACT counts from data/meta.json: train={tr} val={va} vocab={meta['vocab_size']}")
    for s in meta["shards"]:
        nbytes = os.path.getsize(os.path.join("data", s["file"]))
        print(f"  {s['file']} {s['tokens']} toks {nbytes/1024/1024:.1f}MB")
        assert nbytes == 2 * s["tokens"], f"{s['file']}: file size does not match meta (stale meta.json?)"
    assert tr > 400_000_000, f"train tokens {tr} look wrong (expected ~467M)"
    TRAIN_TOKS, VAL_TOKS = tr, va

## Throughput + memory probe (calibrate only)
120 steps at micro_batch_seqs 8/16/32 (rows at 0/50/100, so the skip rate
is measured over steps 51..100, after early GradScaler scale-adjustment).

In [ ]:
import subprocess, threading, time, yaml

def short_cfg(out_path, **overrides):
    """Copy configs/base_30m.yaml with shorter warmup / val interval, for SHORT test runs only.
    (The real warmup is longer than these runs, which trips the 'bad warmup' assertion.)"""
    cfg = yaml.safe_load(open("configs/base_30m.yaml"))
    changed = {}
    def walk(d):
        for k, v in d.items():
            if isinstance(v, dict):
                walk(v)
            elif k in overrides:
                changed[k] = (v, overrides[k])
                d[k] = overrides[k]
    walk(cfg)
    assert "warmup_steps" in changed, "no 'warmup_steps' key found in configs/base_30m.yaml"
    yaml.safe_dump(cfg, open(out_path, "w"))
    print(out_path + ": " + ", ".join(f"{k} {a}->{b}" for k, (a, b) in changed.items()))
    return out_path

PROBE = {}
if MODE != "calibrate":
    print("skip: probe runs in calibrate mode")
else:
    def _poll(stop, rec):
        mx = 0
        while not stop.is_set():
            try:
                q = subprocess.check_output(
                    ["nvidia-smi", "--query-gpu=memory.used",
                     "--format=csv,noheader,nounits"], text=True)
                mx = max(mx, int(q.strip().split()[0]))
            except Exception:
                pass
            time.sleep(2)
        rec.append(mx)
    for MB in (8, 16, 32):
        cfg = short_cfg(f"/tmp/probe_{MB}.yaml", warmup_steps=30)
        stop, rec = threading.Event(), []
        th = threading.Thread(target=_poll, args=(stop, rec))
        th.start()
        r = subprocess.run(
            ["python", "-m", "train.pretrain", "--config", cfg,
             "--resume", "none", "--micro_batch_seqs", str(MB),
             "--max_steps", "120", "--ckpt_dir", f"/tmp/probe_{MB}"],
            capture_output=True, text=True)
        stop.set(); th.join()
        print(r.stdout[-800:] if r.returncode == 0 else "PROBE CRASHED:\n" + r.stderr[-2000:])
        PROBE[MB] = rec[0] if rec else -1
        print(f"micro_batch {MB}: peak GPU mem ~= {PROBE[MB]} MB")

## Shakedown summary (calibrate only): tok/s table + skip-rate gate (fail >= 1%)

In [ ]:
import json
PROBE_TOKS = None
if MODE != "calibrate":
    print("skip: summary runs in calibrate mode")
else:
    passing, rows = [], []
    for MB in (8, 16, 32):
        try:
            recs = [json.loads(l) for l in open(f"/tmp/probe_{MB}/metrics.jsonl")]
        except FileNotFoundError:
            recs = []
        if not recs:
            print(f"mb={MB:2d} no metrics logged (the probe crashed or ran out of memory) [FAIL]")
            continue
        toks = sum(r["tok_s"] for r in recs) / len(recs)
        post = [r for r in recs if r["step"] >= 30]  # past early scaler adjustment
        if len(post) >= 2:
            rate = (post[-1]["skips"] - post[0]["skips"]) / (post[-1]["step"] - post[0]["step"])
        else:
            rate = recs[-1]["skips"] / 120  # fallback: whole probe
        gate = "PASS" if rate < 0.01 else "FAIL"
        rows.append((MB, toks, PROBE[MB], rate, gate))
        print(f"mb={MB:2d} tok/s={toks:7.0f} peak_mem={PROBE[MB]}MB skip_rate_post30={rate:.4f} [{gate}]")
        if gate == "PASS":
            passing.append(toks)
    if passing:
        PROBE_TOKS = max(passing)
        print(f"PROBE_TOKS (best passing) = {PROBE_TOKS:.0f}")
    else:
        print("ALL PROBES FAILED the skip gate: do not start the main run.")
        print("Next: lower --lr (try 3e-4), check install/torch build, then re-run calibrate.")

## Budget (calibrate + main)
Computes max_steps from MEASURED tok/s and a 5.5h target — replaces the provisional default.
In main mode, paste PROBE_TOKS from the calibrate output.

In [ ]:
TOK_S = globals().get("PROBE_TOKS") or None  # calibrate fills this; main: paste the number
# TOK_S = 80000  # <-- main mode: uncomment + paste calibrate's PROBE_TOKS if the cell above is empty
TARGET_H, BATCH = 5.5, 65536
MAX_STEPS = None
if not TOK_S:
    print("no measured tok/s yet: run calibrate first, then paste PROBE_TOKS above.")
else:
    budget = TOK_S * TARGET_H * 3600
    passes = budget / TRAIN_TOKS
    MAX_STEPS = int(budget // BATCH)
    print(f"train_tokens={TRAIN_TOKS} val_tokens={VAL_TOKS}")
    print(f"tok/s={TOK_S:.0f} target={TARGET_H}h -> budget={budget/1e9:.2f}B tokens")
    print(f"passes over train data ~= {passes:.2f} -> MAX_STEPS={MAX_STEPS}")

# After calibrate: paste the MAX_STEPS it printed here so EVERY main session uses the
# same value (the cosine schedule depends on it). Leave None while calibrating.
MAX_STEPS_FIXED = 12158  # from calibrate budget cell (PROBE_TOKS=40245, 5.5h)
if MAX_STEPS_FIXED:
    MAX_STEPS = MAX_STEPS_FIXED
    print("using fixed MAX_STEPS =", MAX_STEPS)


## Kill-resume drill (calibrate only)
Both runs share --max_steps 300 so the cosine schedule is IDENTICAL across
the resume. Run 1 is stopped early by --max_hours (a kill would do the same);
run 2 resumes to 300. metrics.jsonl must read as one continuous curve.

In [ ]:
if MODE != "calibrate":
    print("skip: drill runs in calibrate mode")
else:
    import json
    DRILL_CFG = short_cfg("/tmp/drill.yaml", warmup_steps=30)
    def _rows():
        try:
            return [json.loads(l) for l in open("/tmp/drill/metrics.jsonl")]
        except FileNotFoundError:
            return []
    !python -m train.pretrain --config $DRILL_CFG --resume none --max_steps 300 --max_hours 0.07 --ckpt_dir /tmp/drill 2>&1 | tail -n 3
    r1 = _rows()
    last1 = r1[-1]["step"] if r1 else None
    print("run 1 last logged step:", last1)
    if last1 is None or last1 < 50 or last1 >= 299:
        print("WARNING: run 1 should stop between step 50 and 299. Lower --max_hours if it finished, raise it if it stopped too early.")
    !python -m train.pretrain --config $DRILL_CFG --resume latest --max_steps 300 --ckpt_dir /tmp/drill 2>&1 | tail -n 3
    rows = _rows()
    steps = [r["step"] for r in rows]
    print("rows:", len(rows), "| steps strictly increasing:", steps == sorted(set(steps)))
    print("last 6 (step, loss):", [(r["step"], round(r["loss"], 3)) for r in rows[-6:]])
    print("check: one continuous curve, no loss jump at the resume point")

In [ ]:
if MODE != "calibrate":
    print("skip: sweep runs in calibrate mode")
else:
    import subprocess, json
    sweep, sweep_step = {}, {}
    for LR in ("3e-4", "6e-4", "1e-3"):
        cfg = short_cfg(f"/tmp/sweep_{LR}.yaml", warmup_steps=60, val_every=150)
        subprocess.run(["python", "-m", "train.pretrain", "--config", cfg,
                        "--resume", "none", "--lr", LR,
                        "--max_steps", "900", "--ckpt_dir", f"/tmp/sweep_{LR}"],
                       check=True)
        vals = [json.loads(l) for l in open(f"/tmp/sweep_{LR}/metrics.jsonl") if '"val"' in l]
        assert vals, f"no val rows logged for lr={LR}"
        sweep[LR], sweep_step[LR] = vals[-1]["val"], vals[-1]["step"]
        print(f"lr={LR} val={sweep[LR]:.4f} at step {sweep_step[LR]}")
    assert len(set(sweep_step.values())) == 1, f"val steps differ across runs: {sweep_step}"
    print("--- sweep val losses (same step for all) ---")
    for LR, v in sweep.items():
        print(f"  lr={LR}: val={v:.4f}")
    best = min(sweep, key=sweep.get)
    # Close = within 2% of best -> prefer the LOWEST lr (stability over speed).
    close = [LR for LR in sorted(sweep, key=float)
             if (sweep[LR] - sweep[best]) / sweep[best] < 0.02]
    proposed = close[0]
    print(f"PROPOSED_LR = {proposed} (NOT applied - confirm, then paste into the main cell)")

## Main run (main only): resume latest, computed max_steps, clean exit. Then Save Version.

In [ ]:
if MODE != "main":
    print("skip: main run in main mode")
elif not globals().get("MAX_STEPS"):
    print("no MAX_STEPS: run the budget cell first (needs calibrate numbers).")
else:
    import subprocess
    CONFIRMED_LR = 1e-3  # from calibrate sweep; None = config default
    argv = ["python", "-m", "train.pretrain", "--config", "configs/base_30m.yaml",
            "--resume", "latest", "--max_hours", "3", "--max_steps", str(MAX_STEPS)]
    if CONFIRMED_LR:
        argv += ["--lr", str(CONFIRMED_LR)]
    print("LR:", CONFIRMED_LR or "config default (6e-4)")
    print("max_steps:", MAX_STEPS, "| warmup:", 500, "| config micro_batch_seqs: 8")
    subprocess.run(argv, check=False)

## Backup (main only): newest checkpoints -> private HF repo, graceful skip.
HF_TOKEN goes in Kaggle Secrets. Weights-only copies ride along on Save Version.

In [ ]:
import glob, os
if MODE != "main":
    print("skip: backup runs in main mode")
else:
    from kaggle_secrets import UserSecretsClient
    try:
        tok = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception as e:
        tok = None
        print("no HF_TOKEN secret; skipping backup:", e)
    ckpts = sorted(glob.glob("checkpoints/*/ckpt_step*.pt"))
    if tok and ckpts:
        try:
            import importlib
            try:
                hhub = importlib.import_module("huggingface_hub")
            except ImportError:
                !pip install -q huggingface_hub
                hhub = importlib.import_module("huggingface_hub")
            hhub.login(token=tok)
            api = hhub.HfApi()
            repo = api.create_repo("minigpt-stack-ckpts", private=True, exist_ok=True).repo_id
            for f in ckpts[-2:]:
                api.upload_file(path_or_fileobj=f, path_in_repo=os.path.basename(f), repo_id=repo)
            print("backed up to", repo, ckpts[-2:])
        except Exception as e:
            print("backup skipped (retry next session):", e)
    elif not ckpts:
        print("no checkpoints to back up")